# Асинхронный веб-краулер

### Установка зависимостей

In [1]:
%pip install aiohttp aiofiles beautifulsoup4 lxml aiosqlite

Note: you may need to restart the kernel to use updated packages.


### Импорты
Все библиотеки, которые используются в ноутбуке, собраны здесь.

In [33]:
import asyncio
import aiohttp
import logging
import time
import json
import itertools
import re
import aiofiles
import random
import aiosqlite
import textwrap
import csv
import io

from collections import Counter, deque
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urldefrag
from contextlib import asynccontextmanager, suppress
from urllib.robotparser import RobotFileParser
from abc import ABC, abstractmethod
from datetime import datetime


### Настройка логирования

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logger = logging.getLogger("crawler")

## Класс RetryStrategy

In [4]:
class RetryStrategy:
    def __init__(
        self,
        max_retries: int = 3,
        backoff_factor: float = 2.0,
        retry_on: list = None,
        base_delay: float = 1.0,
        retry_limits: dict | None = None,
        base_delays: dict | None = None,
        status_limits: dict | None = None,
        status_delay_multipliers: dict | None = None,
    ):
        self.max_retries = max_retries
        self.backoff_factor = backoff_factor
        self.retry_on = tuple(retry_on or [TransientError, NetworkError])
        self.base_delay = base_delay
        self.retry_limits = retry_limits or {}
        self.base_delays = base_delays or {}
        self.status_limits = {500: 1} if status_limits is None else status_limits
        self.status_delay_multipliers = (
            {429: 3.0} if status_delay_multipliers is None else status_delay_multipliers
        )

        self.errors_by_type: Counter = Counter()
        self.retry_count = 0
        self.retry_time_total = 0.0
        self.successful_retries = 0
        self.permanent_urls: dict[str, str] = {}

    @staticmethod
    def lookup(settings: dict, error: Exception, default):
        for error_type, value in settings.items():
            if isinstance(error, error_type):
                return value
        return default

    def get_limit(self, error: Exception) -> int:
        status = getattr(error, "status", None)
        limit = self.lookup(self.retry_limits, error, self.max_retries)
        if status in self.status_limits:
            limit = min(limit, self.status_limits[status])
        return limit

    def get_delay(self, retry_number: int, error: Exception | None = None) -> float:
        base = self.lookup(self.base_delays, error, self.base_delay)
        multiplier = self.status_delay_multipliers.get(getattr(error, "status", None), 1.0)
        return base * multiplier * self.backoff_factor ** (retry_number - 1)

    def finish_retry(self, started_at: float):
        self.retry_count += 1
        self.retry_time_total += time.monotonic() - started_at

    async def execute_with_retry(self, coro, *args, **kwargs):
        url = next((arg for arg in args if isinstance(arg, str)), "")
        retries: dict[tuple, int] = {}
        attempt = 0
        retry_started: float | None = None
        while True:
            attempt += 1
            try:
                result = await coro(*args, **kwargs)
            except Exception as raw_error:
                if retry_started is not None:
                    self.finish_retry(retry_started)
                error = classify_error(raw_error, url)
                self.errors_by_type[type(error).__name__] += 1
                key = (type(error), error.status)
                retries[key] = retries.get(key, 0) + 1
                can_retry = (
                    isinstance(error, self.retry_on)
                    and attempt <= self.max_retries
                    and retries[key] <= self.get_limit(error)
                )
                if not can_retry:
                    if isinstance(error, PermanentError):
                        self.permanent_urls[url] = str(error)
                    if error is raw_error:
                        raise
                    raise error from raw_error
                delay = self.get_delay(retries[key], error)
                logger.warning(
                    "%s (%s) на попытке %d, повтор через %.2f сек: %s",
                    type(error).__name__, error, attempt, delay, url,
                )
                retry_started = time.monotonic()
                await asyncio.sleep(delay)
            else:
                if retry_started is not None:
                    self.finish_retry(retry_started)
                    self.successful_retries += 1
                return result

    def get_stats(self) -> dict:
        average = self.retry_time_total / self.retry_count if self.retry_count else 0.0
        return {
            "errors_by_type": dict(self.errors_by_type),
            "retries": self.retry_count,
            "successful_retries": self.successful_retries,
            "avg_retry_time": average,
            "permanent_urls": dict(self.permanent_urls),
        }

## Класс CircuitBreaker

In [5]:
class CircuitBreaker:
    def __init__(
        self,
        failure_rate: float = 0.5,
        window_size: int = 10,
        min_requests: int = 5,
        recovery_timeout: float = 30.0,
    ):
        self.failure_rate = failure_rate
        self.window_size = window_size
        self.min_requests = min_requests
        self.recovery_timeout = recovery_timeout
        self.results: dict[str, deque[bool]] = {}
        self.opened_at: dict[str, float] = {}
        self.probing: set[str] = set()

    def get_state(self, domain: str) -> str:
        if domain not in self.opened_at:
            return "closed"
        if time.monotonic() - self.opened_at[domain] >= self.recovery_timeout:
            return "half-open"
        return "open"

    def allow_request(self, domain: str) -> bool:
        state = self.get_state(domain)
        if state == "closed":
            return True
        if state == "half-open" and domain not in self.probing:
            self.probing.add(domain)
            logger.info("Circuit breaker: пробный запрос к %s", domain)
            return True
        return False

    def record_success(self, domain: str):
        if domain in self.opened_at:
            if domain not in self.probing:
                return
            del self.opened_at[domain]
            self.probing.discard(domain)
            self.results[domain] = deque(maxlen=self.window_size)
            logger.info("Circuit breaker: %s снова доступен", domain)
        self.results.setdefault(domain, deque(maxlen=self.window_size)).append(True)

    def record_failure(self, domain: str):
        if domain in self.probing:
            self.probing.discard(domain)
            self.opened_at[domain] = time.monotonic()
            logger.warning("Circuit breaker: пробный запрос к %s не удался, блокировка продлена", domain)
            return
        results = self.results.setdefault(domain, deque(maxlen=self.window_size))
        results.append(False)
        failures = results.count(False)
        if (
            domain not in self.opened_at
            and len(results) >= self.min_requests
            and failures / len(results) >= self.failure_rate
        ):
            self.opened_at[domain] = time.monotonic()
            logger.warning(
                "Circuit breaker: %s заблокирован на %.1f сек (%d ошибок из %d)",
                domain, self.recovery_timeout, failures, len(results),
            )

## Классы ошибок

In [6]:
class CrawlerError(Exception):
    def __init__(self, message: str, url: str = "", status: int | None = None):
        super().__init__(message)
        self.url = url
        self.status = status


class TransientError(CrawlerError):
    pass


class PermanentError(CrawlerError):
    pass


class NetworkError(CrawlerError):
    pass


class ParseError(CrawlerError):
    pass


def classify_error(error: Exception, url: str) -> CrawlerError:
    if isinstance(error, CrawlerError):
        return error
    if isinstance(error, aiohttp.ClientResponseError):
        message = f"HTTP {error.status}"
        if error.status == 429 or error.status >= 500:
            return TransientError(message, url, error.status)
        return PermanentError(message, url, error.status)
    if isinstance(error, asyncio.TimeoutError):
        return TransientError(type(error).__name__, url)
    if isinstance(error, aiohttp.ClientError):
        return NetworkError(type(error).__name__, url)
    return CrawlerError(type(error).__name__, url)

class CircuitOpenError(CrawlerError):
    pass

## Класс HTMLParser

In [7]:
class HTMLParser:
    def __init__(self):
        self.parse_errors: list[ParseError] = []

    def empty_result(self, url: str) -> dict:
        return {
            "url": url,
            "title": "",
            "text": "",
            "links": [],
            "metadata": {},
            "images": [],
            "headings": {},
            "tables": [],
            "lists": [],
        }

    async def parse_html(self, html: str, url: str) -> dict:
        result = self.empty_result(url)
        try:
            soup = await asyncio.to_thread(BeautifulSoup, html, "lxml")
        except Exception as e:
            logger.warning("Не удалось разобрать HTML (%s): %s", type(e).__name__, url)
            self.parse_errors.append(ParseError(type(e).__name__, url))
            return result

        extractors = {
            "metadata": lambda: self.extract_metadata(soup),
            "text": lambda: self.extract_text(soup),
            "links": lambda: self.extract_links(soup, url),
            "images": lambda: self.extract_images(soup, url),
            "headings": lambda: self.extract_headings(soup),
            "tables": lambda: self.extract_tables(soup),
            "lists": lambda: self.extract_lists(soup),
        }
        for field, extract in extractors.items():
            try:
                result[field] = extract()
            except Exception as e:
                logger.warning(
                    "Ошибка при извлечении %s (%s): %s", field, type(e).__name__, url
                )
                self.parse_errors.append(ParseError(f"{field}: {type(e).__name__}", url))

        result["title"] = result["metadata"].get("title", "")
        return result

    def extract_links(self, soup: BeautifulSoup, base_url: str) -> list[str]:
        links = []
        for tag in soup.find_all("a", href=True):
            absolute_url = urljoin(base_url, tag["href"])
            if self.is_valid_url(absolute_url):
                links.append(absolute_url)
        return links

    def is_valid_url(self, url: str) -> bool:
        parsed = urlparse(url)
        return parsed.scheme in ("http", "https") and bool(parsed.netloc)

    def extract_text(self, soup: BeautifulSoup, selector: str = None) -> str:
        root = soup.select_one(selector) if selector else soup
        if root is None:
            return ""
        return root.get_text(separator=" ", strip=True)

    def extract_metadata(self, soup: BeautifulSoup) -> dict:
        def meta_content(name: str) -> str:
            tag = soup.find("meta", attrs={"name": name})
            return tag.get("content", "") if tag else ""

        title = soup.title.get_text(strip=True) if soup.title else ""
        return {
            "title": title,
            "description": meta_content("description"),
            "keywords": meta_content("keywords"),
        }
    
    def extract_images(self, soup: BeautifulSoup, base_url: str) -> list[dict]:
        images = []
        for tag in soup.find_all("img", src=True):
            images.append({
                "src": urljoin(base_url, tag["src"]),
                "alt": tag.get("alt", ""),
            })
        return images

    def extract_headings(self, soup: BeautifulSoup) -> dict[str, list[str]]:
        headings = {}
        for level in ("h1", "h2", "h3"):
            headings[level] = [
                tag.get_text(strip=True) for tag in soup.find_all(level)
            ]
        return headings

    def extract_tables(self, soup: BeautifulSoup) -> list[list[list[str]]]:
        tables = []
        for table in soup.find_all("table"):
            rows = []
            for tr in table.find_all("tr"):
                cells = [
                    cell.get_text(strip=True) for cell in tr.find_all(["th", "td"])
                ]
                if cells:
                    rows.append(cells)
            tables.append(rows)
        return tables

    def extract_lists(self, soup: BeautifulSoup) -> list[dict]:
        lists = []
        for tag in soup.find_all(["ul", "ol"]):
            items = [
                li.get_text(strip=True) for li in tag.find_all("li", recursive=False)
            ]
            lists.append({"type": tag.name, "items": items})
        return lists

## Класс CrawlerQueue

In [8]:
class CrawlerQueue:
    def __init__(self):
        self.queue = asyncio.PriorityQueue()
        self.counter = itertools.count()
        self.seen: set[str] = set()
        self.processed: set[str] = set()
        self.failed: dict[str, str] = {}

    def add_url(self, url: str, priority: int = 0):
        if url in self.seen:
            return
        self.seen.add(url)
        self.queue.put_nowait((-priority, next(self.counter), url))

    def requeue(self, url: str, priority: int = 0):
        self.queue.put_nowait((-priority, next(self.counter), url))

    async def get_next(self) -> str | None:
        if self.queue.empty():
            return None
        _, _, url = await self.queue.get()
        return url

    def mark_processed(self, url: str):
        self.processed.add(url)

    def mark_failed(self, url: str, error: str):
        self.failed[url] = error

    def get_stats(self) -> dict:
        return {
            "queued": self.queue.qsize(),
            "processed": len(self.processed),
            "failed": len(self.failed),
            "total_seen": len(self.seen),
        }

## Класс SemaphoreManager

In [9]:
class SemaphoreManager:
    def __init__(self, global_limit: int = 10, per_domain_limit: int = 2):
        self.global_semaphore = asyncio.Semaphore(global_limit)
        self.per_domain_limit = per_domain_limit
        self.domain_semaphores: dict[str, asyncio.Semaphore] = {}
        self.active_total = 0
        self.active_by_domain: dict[str, int] = {}

    @asynccontextmanager
    async def acquire(self, url: str):
        domain = urlparse(url).netloc
        if domain not in self.domain_semaphores:
            self.domain_semaphores[domain] = asyncio.Semaphore(self.per_domain_limit)

        async with self.domain_semaphores[domain]:
            async with self.global_semaphore:
                self.active_total += 1
                self.active_by_domain[domain] = self.active_by_domain.get(domain, 0) + 1
                try:
                    yield
                finally:
                    self.active_total -= 1
                    self.active_by_domain[domain] -= 1

    def get_active(self) -> dict:
        return {
            "total": self.active_total,
            "by_domain": {
                domain: count
                for domain, count in self.active_by_domain.items()
                if count > 0
            },
        }

## Класс RateLimiter

In [10]:
class RateLimiter:
    def __init__(
        self,
        requests_per_second: float = 1.0,
        per_domain: bool = True,
        min_delay: float = 0.0,
        jitter: float = 0.0,
        stats_window: float = 5.0,
    ):
        self.interval = max(1.0 / requests_per_second, min_delay)
        self.jitter = jitter
        self.per_domain = per_domain
        self.next_allowed: dict[str, float] = {}
        self.domain_intervals: dict[str, float] = {}

        self.stats_window = stats_window
        self.recent_requests: deque[float] = deque()
        self.first_request: float | None = None
        self.last_request: dict[str, float] = {}
        self.waiting_domains: set[str] = set()
        self.gaps_total = 0.0
        self.gaps_count = 0
        self.min_gap: float | None = None

    def set_domain_delay(self, domain: str, delay: float):
        self.domain_intervals[domain] = max(self.interval, delay)

    def time_until_allowed(self, domain: str = None) -> float:
        key = domain if self.per_domain and domain else "global"
        return max(0.0, self.next_allowed.get(key, 0.0) - time.monotonic())

    def mark_waiting(self, domain: str):
        self.waiting_domains.add(domain)

    async def acquire(self, domain: str = None):
        key = domain if self.per_domain and domain else "global"
        interval = self.domain_intervals.get(key, self.interval)
        now = time.monotonic()
        slot = max(now, self.next_allowed.get(key, now))
        self.next_allowed[key] = slot + interval + random.uniform(0, self.jitter)
        wait = slot - now
        if wait > 0:
            await asyncio.sleep(wait)
        self.record_request(domain or "global", throttled=wait > 0)

    def record_request(self, domain: str, throttled: bool):
        now = time.monotonic()
        if self.first_request is None:
            self.first_request = now
        self.recent_requests.append(now)
        throttled = throttled or domain in self.waiting_domains
        self.waiting_domains.discard(domain)
        if throttled and domain in self.last_request:
            gap = now - self.last_request[domain]
            self.gaps_total += gap
            self.gaps_count += 1
            if self.min_gap is None or gap < self.min_gap:
                self.min_gap = gap
        self.last_request[domain] = now

    def get_stats(self) -> dict:
        now = time.monotonic()
        while self.recent_requests and now - self.recent_requests[0] > self.stats_window:
            self.recent_requests.popleft()
        if self.first_request is None:
            current_rate = 0.0
        else:
            window = max(min(self.stats_window, now - self.first_request), 1.0)
            current_rate = len(self.recent_requests) / window
        avg_delay = self.gaps_total / self.gaps_count if self.gaps_count else 0.0
        return {
            "requests_per_second": current_rate,
            "avg_delay": avg_delay,
            "min_delay": self.min_gap or 0.0,
        }

## Класс RobotsParser

In [11]:
class RobotsParser:
    def __init__(self, session: aiohttp.ClientSession):
        self.session = session
        self.rules: dict[str, RobotFileParser] = {}
        self.crawl_delays: dict[str, dict[str, float]] = {}
        self.locks: dict[str, asyncio.Lock] = {}

    async def fetch_robots(self, base_url: str) -> dict:
        parsed = urlparse(base_url)
        domain = parsed.netloc
        robots_url = f"{parsed.scheme}://{domain}/robots.txt"

        if domain not in self.locks:
            self.locks[domain] = asyncio.Lock()
        async with self.locks[domain]:
            if domain not in self.rules:
                parser = RobotFileParser(robots_url)
                delays: dict[str, float] = {}
                try:
                    async with self.session.get(robots_url) as response:
                        if response.status == 200:
                            text = await response.text()
                            parser.parse(text.splitlines())
                            delays = self.parse_crawl_delays(text)
                        elif response.status in (401, 403):
                            parser.disallow_all = True
                        else:
                            parser.allow_all = True
                except (aiohttp.ClientError, asyncio.TimeoutError) as e:
                    logger.warning("Не удалось загрузить robots.txt (%s): %s", type(e).__name__, robots_url)
                    parser.allow_all = True
                self.rules[domain] = parser
                self.crawl_delays[domain] = delays

        parser = self.rules[domain]
        return {
            "domain": domain,
            "robots_url": robots_url,
            "allow_all": parser.allow_all,
            "disallow_all": parser.disallow_all,
            "crawl_delay": self.get_crawl_delay(domain=domain),
        }

    def parse_crawl_delays(self, text: str) -> dict[str, float]:
        delays: dict[str, float] = {}
        agents: list[str] = []
        in_rules = False
        for raw_line in text.splitlines():
            line = raw_line.split("#", 1)[0].strip()
            if ":" not in line:
                continue
            field, value = line.split(":", 1)
            field = field.strip().lower()
            value = value.strip()
            if field == "user-agent":
                if in_rules:
                    agents = []
                    in_rules = False
                agents.append(value.lower())
                continue
            in_rules = True
            if field == "crawl-delay":
                try:
                    delay = float(value)
                except ValueError:
                    continue
                for agent in agents:
                    delays[agent] = delay
        return delays

    def can_fetch(self, url: str, user_agent: str = "*") -> bool:
        parser = self.rules.get(urlparse(url).netloc)
        if parser is None:
            return True
        return parser.can_fetch(user_agent, url)

    def get_crawl_delay(self, user_agent: str = "*", domain: str | None = None) -> float:
        delays = self.crawl_delays.get(domain, {})
        name = user_agent.split("/")[0].lower()
        for agent, delay in delays.items():
            if agent != "*" and agent in name:
                return delay
        return delays.get("*", 0.0)

## Класс DataStorage

In [38]:
class DataStorage(ABC):
    FIELDS = [
        "url", "title", "text", "links", "metadata",
        "crawled_at", "status_code", "content_type",
    ]

    @staticmethod
    def normalize(data: dict) -> dict:
        return {
            "url": str(data.get("url", "")),
            "title": data.get("title") or "",
            "text": data.get("text") or "",
            "links": list(data.get("links") or []),
            "metadata": dict(data.get("metadata") or {}),
            "crawled_at": data.get("crawled_at") or datetime.now(),
            "status_code": data.get("status_code"),
            "content_type": data.get("content_type") or "",
        }

    @abstractmethod
    async def save(self, data: dict):
        pass

    @abstractmethod
    async def close(self):
        pass

## Класс JSONStorage

In [39]:
class JSONStorage(DataStorage):
    def __init__(self, path: str, pretty: bool = False, encoding: str = "utf-8"):
        self.path = path
        self.pretty = pretty
        self.encoding = encoding
        self.file = None
        self.count = 0
        self.closed = False
        self.lock = asyncio.Lock()

    async def open(self):
        self.file = await aiofiles.open(self.path, "w", encoding=self.encoding)
        if self.pretty:
            await self.file.write("[\n")

    def serialize(self, data: dict) -> str:
        if self.pretty:
            text = json.dumps(data, ensure_ascii=False, indent=2, default=str)
            return textwrap.indent(text, "  ")
        return json.dumps(data, ensure_ascii=False, default=str)

    async def save(self, data: dict):
        data = self.normalize(data)
        async with self.lock:
            if self.file is None:
                await self.open()
            text = self.serialize(data)
            if self.pretty:
                separator = ",\n" if self.count else ""
                await self.file.write(separator + text)
            else:
                await self.file.write(text + "\n")
            self.count += 1

    async def close(self):
        async with self.lock:
            if self.closed:
                return
            if self.file is None:
                await self.open()
            if self.pretty:
                await self.file.write("\n]\n")
            await self.file.close()
            self.closed = True

## Класс CSVStorage

In [34]:
class CSVStorage(DataStorage):
    def __init__(self, path: str, encoding: str = "utf-8-sig", delimiter: str = ","):
        self.path = path
        self.encoding = encoding
        self.delimiter = delimiter
        self.fieldnames: list[str] | None = None
        self.file = None
        self.closed = False
        self.lock = asyncio.Lock()

    @staticmethod
    def to_cell(value) -> str:
        if value is None:
            return ""
        if isinstance(value, (list, dict)):
            return json.dumps(value, ensure_ascii=False, default=str)
        return str(value)

    def format_rows(self, rows: list[list[str]]) -> str:
        buffer = io.StringIO()
        csv.writer(buffer, delimiter=self.delimiter).writerows(rows)
        return buffer.getvalue()

    async def save(self, data: dict):
        async with self.lock:
            rows = []
            if self.file is None:
                self.file = await aiofiles.open(
                    self.path, "w", encoding=self.encoding, errors="replace", newline=""
                )
                self.fieldnames = list(data.keys())
                rows.append(self.fieldnames)
            rows.append([self.to_cell(data.get(field)) for field in self.fieldnames])
            await self.file.write(self.format_rows(rows))

    async def close(self):
        async with self.lock:
            if self.closed:
                return
            if self.file is not None:
                await self.file.close()
            self.closed = True

## Класс SQLiteStorage

In [40]:
class SQLiteStorage(DataStorage):
    COLUMNS = DataStorage.FIELDS

    def __init__(self, path: str, batch_size: int = 50):
        self.path = path
        self.batch_size = batch_size
        self.db: aiosqlite.Connection | None = None
        self.buffer: list[tuple] = []
        self.batches_written = 0
        self.closed = False
        self.lock = asyncio.Lock()

    async def init_db(self):
        self.db = await aiosqlite.connect(self.path)
        await self.db.execute(
            """
            CREATE TABLE IF NOT EXISTS pages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                url TEXT NOT NULL UNIQUE,
                title TEXT,
                text TEXT,
                links TEXT,
                metadata TEXT,
                crawled_at TEXT,
                status_code INTEGER,
                content_type TEXT
            )
            """
        )
        await self.db.execute("CREATE INDEX IF NOT EXISTS idx_pages_status ON pages(status_code)")
        await self.db.execute("CREATE INDEX IF NOT EXISTS idx_pages_crawled_at ON pages(crawled_at)")
        await self.db.commit()

    @staticmethod
    def to_value(value):
        if isinstance(value, (list, dict)):
            return json.dumps(value, ensure_ascii=False, default=str)
        if isinstance(value, datetime):
            return value.isoformat()
        return value

    async def flush(self):
        if not self.buffer:
            return
        placeholders = ", ".join("?" for _ in self.COLUMNS)
        await self.db.executemany(
            f"INSERT OR REPLACE INTO pages ({', '.join(self.COLUMNS)}) VALUES ({placeholders})",
            self.buffer,
        )
        await self.db.commit()
        self.buffer.clear()
        self.batches_written += 1

    async def save(self, data: dict):
        async with self.lock:
            if self.db is None:
                await self.init_db()
            self.buffer.append(tuple(self.to_value(data.get(column)) for column in self.COLUMNS))
            if len(self.buffer) >= self.batch_size:
                await self.flush()

    async def close(self):
        async with self.lock:
            if self.closed:
                return
            if self.db is None:
                await self.init_db()
            await self.flush()
            await self.db.close()
            self.closed = True

In [41]:
poor = {"url": "https://example.com"}
rich = {
    "url": "https://example.com/page",
    "title": "Страница",
    "text": "Текст",
    "links": ["https://example.com/a"],
    "metadata": {"description": "описание"},
    "images": [{"src": "logo.png", "alt": ""}],
    "tables": [],
    "status_code": 200,
    "content_type": "text/html; charset=utf-8",
}

for record in (poor, rich):
    normalized = DataStorage.normalize(record)
    print(list(normalized) == DataStorage.FIELDS)
    print(normalized, "\n")

True
{'url': 'https://example.com', 'title': '', 'text': '', 'links': [], 'metadata': {}, 'crawled_at': datetime.datetime(2026, 10, 23, 16, 1, 59, 904564), 'status_code': None, 'content_type': ''} 

True
{'url': 'https://example.com/page', 'title': 'Страница', 'text': 'Текст', 'links': ['https://example.com/a'], 'metadata': {'description': 'описание'}, 'crawled_at': datetime.datetime(2026, 10, 23, 16, 1, 59, 904980), 'status_code': 200, 'content_type': 'text/html; charset=utf-8'} 



## Класс AsyncCrawler

In [42]:
class AsyncCrawler:
    def __init__(
        self,
        max_concurrent: int = 10,
        max_depth: int = 2,
        per_domain_limit: int | None = None,
        requests_per_second: float | None = None,
        respect_robots: bool = False,
        user_agent: str = "AsyncCrawler/1.0",
        min_delay: float = 0.0,
        jitter: float = 0.0,
        max_retries: int = 0,
        backoff_base: float = 1.0,
        retry_strategy: RetryStrategy | None = None,
        circuit_breaker: CircuitBreaker | None = None,
        storage: DataStorage | None = None,
        connect_timeout: float = 5,
        read_timeout: float = 10,
        total_timeout: float | None = None,
        timeout_growth: float = 1.5,
    ):
        self.max_concurrent = max_concurrent
        self.max_depth = max_depth
        self.respect_robots = respect_robots
        self.user_agent = user_agent
        self.jitter = jitter
        self.connect_timeout = connect_timeout
        self.read_timeout = read_timeout
        self.total_timeout = total_timeout
        self.timeout_growth = timeout_growth
        self.circuit_breaker = circuit_breaker
        self.storage = storage
        self.retry_strategy = retry_strategy or RetryStrategy(
            max_retries=max_retries,
            base_delay=backoff_base,
        )
        self.semaphores = SemaphoreManager(
            global_limit=max_concurrent,
            per_domain_limit=per_domain_limit or max_concurrent,
        )
        self.parser = HTMLParser()
        self.rate_limiter = RateLimiter(
            requests_per_second=requests_per_second or float("inf"),
            min_delay=min_delay,
            jitter=jitter,
        )

        self.queue = CrawlerQueue()
        self.visited_urls: set[str] = set()
        self.failed_urls: dict[str, str] = {}
        self.processed_urls: dict[str, dict] = {}
        self.blocked_urls: set[str] = set()
        self.url_depths: dict[str, int] = {}
        self.attempts: dict[str, int] = {}
        self.responses: dict[str, dict] = {}
        self.in_progress = 0
        self.same_domain_only = False
        self.allowed_domains: set[str] = set()
        self.exclude_patterns: list[re.Pattern] = []
        self.include_patterns: list[re.Pattern] = []

        # используется системный поиск адресов, чтобы краулер работал через VPN
        connector = aiohttp.TCPConnector(
            limit=max_concurrent,
            resolver=aiohttp.ThreadedResolver(),
        )
        self.session = aiohttp.ClientSession(
            timeout=self.make_timeout(attempt=1),
            connector=connector,
            headers={"User-Agent": user_agent},
        )
        self.robots = RobotsParser(self.session)

    def make_timeout(self, attempt: int) -> aiohttp.ClientTimeout:
        factor = self.timeout_growth ** (attempt - 1)
        return aiohttp.ClientTimeout(
            total=self.total_timeout * factor if self.total_timeout else None,
            connect=self.connect_timeout * factor,
            sock_read=self.read_timeout * factor,
        )

    async def check_robots(self, url: str) -> bool:
        domain = urlparse(url).netloc
        await self.robots.fetch_robots(url)
        delay = self.robots.get_crawl_delay(self.user_agent, domain)
        if delay:
            self.rate_limiter.set_domain_delay(domain, delay)
        if self.robots.can_fetch(url, self.user_agent):
            return True
        logger.warning("Заблокировано robots.txt: %s", url)
        self.blocked_urls.add(url)
        return False

    async def fetch_url(self, url: str) -> str | None:
        if self.respect_robots and not await self.check_robots(url):
            return None
        self.attempts[url] = 0
        try:
            text = await self.retry_strategy.execute_with_retry(self.try_fetch, url)
        except CrawlerError as error:
            logger.error(
                "Итог: неудача после %d попыток (%s: %s): %s",
                self.attempts[url], type(error).__name__, error, url,
            )
            self.failed_urls[url] = f"{type(error).__name__}: {error}"
            return None
        logger.info("Итог: успех с попытки %d: %s", self.attempts[url], url)
        return text

    async def try_fetch(self, url: str) -> str:
        self.attempts[url] = self.attempts.get(url, 0) + 1
        attempt = self.attempts[url]
        domain = urlparse(url).netloc
        if self.circuit_breaker and not self.circuit_breaker.allow_request(domain):
            logger.warning("Попытка %d: домен %s временно заблокирован: %s", attempt, domain, url)
            raise CircuitOpenError(f"домен {domain} временно заблокирован", url)

        timeout = self.make_timeout(attempt)
        async with self.semaphores.acquire(url):
            await self.rate_limiter.acquire(domain)
            logger.info(
                "Начало загрузки (попытка %d, таймаут чтения %.1f сек): %s",
                attempt, timeout.sock_read, url,
            )
            try:
                async with self.session.get(url, timeout=timeout) as response:
                    response.raise_for_status()
                    text = await response.text()
            except Exception as raw_error:
                error = classify_error(raw_error, url)
                logger.error(
                    "Попытка %d: %s (%s): %s",
                    attempt, type(error).__name__, error, url,
                )
                if self.circuit_breaker:
                    if isinstance(error, (TransientError, NetworkError)):
                        self.circuit_breaker.record_failure(domain)
                    else:
                        self.circuit_breaker.record_success(domain)
                if error is raw_error:
                    raise
                raise error from raw_error
            if self.circuit_breaker:
                self.circuit_breaker.record_success(domain)
            self.responses[url] = {
                "status_code": response.status,
                "content_type": response.headers.get("Content-Type", ""),
            }
            logger.info(
                "Попытка %d успешна: %s (статус %s, %d символов)",
                attempt, url, response.status, len(text),
            )
            return text

    async def fetch_urls(self, urls: list[str]) -> dict[str, str]:
        tasks = [self.fetch_url(url) for url in urls]
        pages = await asyncio.gather(*tasks)
        return {url: page for url, page in zip(urls, pages) if page is not None}

    async def fetch_and_parse(self, url: str) -> dict:
        html = await self.fetch_url(url)
        if html is None:
            return self.parser.empty_result(url)
        return await self.parser.parse_html(html, url)

    async def crawl(
        self,
        start_urls: list[str],
        max_pages: int = 100,
        same_domain_only: bool = False,
        exclude_patterns: list[str] | None = None,
        include_patterns: list[str] | None = None,
    ) -> dict[str, dict]:
        self.queue = CrawlerQueue()
        self.visited_urls = set()
        self.failed_urls = {}
        self.processed_urls = {}
        self.blocked_urls = set()
        self.url_depths = {}
        self.attempts = {}
        self.responses = {}
        self.in_progress = 0

        self.same_domain_only = same_domain_only
        self.allowed_domains = {urlparse(url).netloc for url in start_urls}
        self.exclude_patterns = [re.compile(p) for p in exclude_patterns or []]
        self.include_patterns = [re.compile(p) for p in include_patterns or []]

        for url in start_urls:
            url, _ = urldefrag(url)
            self.url_depths[url] = 0
            self.queue.add_url(url, priority=0)

        start_time = time.perf_counter()
        progress = asyncio.create_task(self.report_progress(start_time))
        workers = [
            asyncio.create_task(self.worker(max_pages))
            for _ in range(self.max_concurrent)
        ]
        try:
            await asyncio.gather(*workers)
        finally:
            progress.cancel()
            with suppress(asyncio.CancelledError):
                await progress
            self.print_progress(start_time)
        return self.processed_urls

    def is_allowed(self, url: str) -> bool:
        if self.same_domain_only and urlparse(url).netloc not in self.allowed_domains:
            return False
        if any(pattern.search(url) for pattern in self.exclude_patterns):
            return False
        if self.include_patterns and not any(
            pattern.search(url) for pattern in self.include_patterns
        ):
            return False
        return True

    async def report_progress(self, start_time: float, interval: float = 1.0):
        while True:
            await asyncio.sleep(interval)
            self.print_progress(start_time)

    def print_progress(self, start_time: float):
        stats = self.queue.get_stats()
        rate_stats = self.rate_limiter.get_stats()
        elapsed = time.perf_counter() - start_time
        speed = stats["processed"] / elapsed if elapsed > 0 else 0
        print(
            f"Обработано: {stats['processed']} | "
            f"В очереди: {stats['queued']} | "
            f"Ошибок: {stats['failed']} | "
            f"Заблокировано: {len(self.blocked_urls)} | "
            f"Скорость: {speed:.1f} стр/сек | "
            f"Запросов: {rate_stats['requests_per_second']:.1f} req/sec | "
            f"Средняя задержка: {rate_stats['avg_delay']:.2f} сек"
        )

    async def worker(self, max_pages: int):
        while len(self.visited_urls) < max_pages:
            url = await self.queue.get_next()
            if url is None:
                if self.in_progress == 0:
                    break
                await asyncio.sleep(0.1)
                continue

            domain = urlparse(url).netloc
            wait = self.rate_limiter.time_until_allowed(domain)
            if wait > 0:
                self.rate_limiter.mark_waiting(domain)
                self.queue.requeue(url, priority=-self.url_depths[url])
                await asyncio.sleep(min(wait, 0.05))
                continue

            self.visited_urls.add(url)
            self.in_progress += 1
            try:
                await self.process_url(url)
            finally:
                self.in_progress -= 1

    async def process_url(self, url: str):
        page = await self.fetch_and_parse(url)

        if url in self.blocked_urls:
            return
        if url in self.failed_urls:
            self.queue.mark_failed(url, self.failed_urls[url])
            return

        self.processed_urls[url] = page
        self.queue.mark_processed(url)
        if self.storage:
            record = {**page, **self.responses.get(url, {}), "crawled_at": datetime.now()}
            await self.storage.save(record)
        self.expand_links(url)

    def expand_links(self, url: str):
        depth = self.url_depths[url]
        if depth >= self.max_depth:
            return
        for link in self.processed_urls[url]["links"]:
            link, _ = urldefrag(link)
            if self.is_allowed(link):
                self.add_link(link, depth + 1)

    def add_link(self, link: str, depth: int):
        known_depth = self.url_depths.get(link)
        if known_depth is not None and known_depth <= depth:
            return
        self.url_depths[link] = depth
        if link in self.processed_urls:
            self.expand_links(link)
        else:
            self.queue.add_url(link, priority=-depth)

    def get_error_stats(self) -> dict:
        stats = self.retry_strategy.get_stats()
        errors_by_type = Counter(stats["errors_by_type"])
        if self.parser.parse_errors:
            errors_by_type["ParseError"] += len(self.parser.parse_errors)
        stats["errors_by_type"] = dict(errors_by_type)
        return stats

    async def close(self):
        if self.storage:
            await self.storage.close()
        await self.session.close()

## Демонстрация 1 дня

In [13]:
DEMO_URLS = [
    "https://example.com",
    "https://www.python.org",
    "https://httpbin.org/get",
    "https://httpbin.org/html",
    "https://httpbin.org/delay/1",
    "https://httpbin.org/delay/2",
    "https://httpbin.org/delay/3",
    "https://httpbin.org/status/404",
]


async def run_sequential(urls: list[str]) -> tuple[dict[str, str], float]:
    crawler = AsyncCrawler(max_concurrent=5)
    results = {}
    start = time.perf_counter()
    try:
        for url in urls:
            page = await crawler.fetch_url(url)
            if page is not None:
                results[url] = page
    finally:
        await crawler.close()
    return results, time.perf_counter() - start


async def run_parallel(urls: list[str]) -> tuple[dict[str, str], float]:
    crawler = AsyncCrawler(max_concurrent=5)
    start = time.perf_counter()
    try:
        results = await crawler.fetch_urls(urls)
    finally:
        await crawler.close()
    return results, time.perf_counter() - start


def print_report(title: str, urls: list[str], results: dict[str, str], elapsed: float):
    print(f"\n{title}")
    for url in urls:
        mark = "✅" if url in results else "❌"
        print(f"  {mark} {url}")
    print(f"  Загружено {len(results)} из {len(urls)} за {elapsed:.2f} сек")


async def main():
    seq_results, seq_time = await run_sequential(DEMO_URLS)
    par_results, par_time = await run_parallel(DEMO_URLS)

    print_report("Последовательная загрузка", DEMO_URLS, seq_results, seq_time)
    print_report("Параллельная загрузка", DEMO_URLS, par_results, par_time)
    print(f"\nПараллельно быстрее в {seq_time / par_time:.1f} раза")


await main()

11:49:05 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://example.com
11:49:07 [INFO] Попытка 1 успешна: https://example.com (статус 200, 577 символов)
11:49:07 [INFO] Итог: успех с попытки 1: https://example.com
11:49:07 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://www.python.org
11:49:07 [INFO] Попытка 1 успешна: https://www.python.org (статус 200, 52743 символов)
11:49:07 [INFO] Итог: успех с попытки 1: https://www.python.org
11:49:07 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://httpbin.org/get
11:49:08 [INFO] Попытка 1 успешна: https://httpbin.org/get (статус 200, 312 символов)
11:49:08 [INFO] Итог: успех с попытки 1: https://httpbin.org/get
11:49:08 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://httpbin.org/html
11:49:08 [INFO] Попытка 1 успешна: https://httpbin.org/html (статус 200, 3739 символов)
11:49:08 [INFO] Итог: успех с попытки 1: https://httpbin.org/html
11:49:08 [INFO] Начало загруз


Последовательная загрузка
  ✅ https://example.com
  ✅ https://www.python.org
  ✅ https://httpbin.org/get
  ✅ https://httpbin.org/html
  ✅ https://httpbin.org/delay/1
  ✅ https://httpbin.org/delay/2
  ✅ https://httpbin.org/delay/3
  ❌ https://httpbin.org/status/404
  Загружено 7 из 8 за 10.29 сек

Параллельная загрузка
  ✅ https://example.com
  ✅ https://www.python.org
  ✅ https://httpbin.org/get
  ✅ https://httpbin.org/html
  ✅ https://httpbin.org/delay/1
  ✅ https://httpbin.org/delay/2
  ✅ https://httpbin.org/delay/3
  ❌ https://httpbin.org/status/404
  Загружено 7 из 8 за 4.09 сек

Параллельно быстрее в 2.5 раза


## Тестирование 1 дня

In [14]:
async def test_valid_urls():
    crawler = AsyncCrawler()
    try:
        results = await crawler.fetch_urls([
            "https://example.com",
            "https://httpbin.org/html",
        ])
    finally:
        await crawler.close()
    assert len(results) == 2, f"ожидали 2 страницы, получили {len(results)}"
    assert "Example Domain" in results["https://example.com"], "не тот текст страницы"


async def test_invalid_urls():
    crawler = AsyncCrawler()
    try:
        results = await crawler.fetch_urls([
            "https://this-site-does-not-exist-12345.com",
            "https://httpbin.org/status/404",
        ])
    finally:
        await crawler.close()
    assert results == {}, f"ожидали пустой результат, получили {list(results)}"


async def test_timeout():
    crawler = AsyncCrawler(read_timeout=1)
    start = time.perf_counter()
    try:
        page = await crawler.fetch_url("https://httpbin.org/delay/5")
    finally:
        await crawler.close()
    elapsed = time.perf_counter() - start
    assert page is None, "страница не должна была загрузиться"
    assert elapsed < 4, f"таймаут не сработал, ждали {elapsed:.1f} сек"


async def test_parallel_faster():
    urls = [
        "https://httpbin.org/delay/1",
        "https://httpbin.org/delay/2",
        "https://httpbin.org/delay/3",
    ]
    _, seq_time = await run_sequential(urls)
    _, par_time = await run_parallel(urls)
    assert par_time < seq_time * 0.75, (
        f"параллельно {par_time:.1f} сек, последовательно {seq_time:.1f} сек"
    )


async def run_tests(tests: list):
    logger.setLevel(logging.WARNING)
    passed = 0
    try:
        for test in tests:
            try:
                await test()
                print(f"✅ {test.__name__}")
                passed += 1
            except AssertionError as e:
                print(f"❌ {test.__name__}: {e}")
    finally:
        logger.setLevel(logging.INFO)
    print(f"\nПройдено {passed} из {len(tests)}")


await run_tests([test_valid_urls, test_invalid_urls, test_timeout, test_parallel_faster])

11:49:21 [ERROR] Попытка 1: NetworkError (ClientConnectorDNSError): https://this-site-does-not-exist-12345.com
11:49:21 [ERROR] Итог: неудача после 1 попыток (NetworkError: ClientConnectorDNSError): https://this-site-does-not-exist-12345.com


✅ test_valid_urls


11:49:21 [ERROR] Попытка 1: PermanentError (HTTP 404): https://httpbin.org/status/404
11:49:21 [ERROR] Итог: неудача после 1 попыток (PermanentError: HTTP 404): https://httpbin.org/status/404


✅ test_invalid_urls


11:49:23 [ERROR] Попытка 1: TransientError (SocketTimeoutError): https://httpbin.org/delay/5
11:49:23 [ERROR] Итог: неудача после 1 попыток (TransientError: SocketTimeoutError): https://httpbin.org/delay/5


✅ test_timeout
✅ test_parallel_faster

Пройдено 4 из 4


## Демонстрация 2 дня

In [15]:
DEMO_PAGES = [
    "https://example.com",
    "https://www.python.org",
    "https://httpbin.org/html",
    "https://docs.python.org/3/",
]


def summarize(page: dict) -> dict:
    return {
        "url": page["url"],
        "title": page["title"],
        "h1": page["headings"].get("h1", []),
        "text_length": len(page["text"]),
        "links_count": len(page["links"]),
        "links": page["links"][:5],
        "images_count": len(page["images"]),
    }


async def demo_parsing(urls: list[str]):
    crawler = AsyncCrawler(max_concurrent=5)
    start = time.perf_counter()
    try:
        pages = await asyncio.gather(*(crawler.fetch_and_parse(url) for url in urls))
    finally:
        await crawler.close()
    elapsed = time.perf_counter() - start

    for page in pages:
        print(json.dumps(summarize(page), ensure_ascii=False, indent=2))

    total_links = sum(len(page["links"]) for page in pages)
    total_text = sum(len(page["text"]) for page in pages)
    print("\nСтатистика")
    print(f"  Страниц: {len(pages)}")
    print(f"  Всего ссылок: {total_links}")
    print(f"  Общая длина текста: {total_text} символов")
    print(f"  Время: {elapsed:.2f} сек")


await demo_parsing(DEMO_PAGES)

11:49:34 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://example.com
11:49:34 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://www.python.org
11:49:34 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://httpbin.org/html
11:49:34 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://docs.python.org/3/
11:49:34 [INFO] Попытка 1 успешна: https://example.com (статус 200, 577 символов)
11:49:34 [INFO] Итог: успех с попытки 1: https://example.com
11:49:34 [INFO] Попытка 1 успешна: https://docs.python.org/3/ (статус 200, 19620 символов)
11:49:34 [INFO] Итог: успех с попытки 1: https://docs.python.org/3/
11:49:34 [INFO] Попытка 1 успешна: https://www.python.org (статус 200, 52743 символов)
11:49:34 [INFO] Итог: успех с попытки 1: https://www.python.org
11:49:35 [INFO] Попытка 1 успешна: https://httpbin.org/html (статус 200, 3739 символов)
11:49:35 [INFO] Итог: успех с попытки 1: https://httpbin.org/html


{
  "url": "https://example.com",
  "title": "Example Domain",
  "h1": [],
  "text_length": 171,
  "links_count": 0,
  "links": [],
  "images_count": 0
}
{
  "url": "https://www.python.org",
  "title": "Welcome to Python.org",
  "h1": [
    "Functions Defined",
    "All the Flow You’d Expect",
    "Compound Data Types",
    "Intuitive Interpretation",
    "Quick & Easy to Learn"
  ],
  "text_length": 6615,
  "links_count": 213,
  "links": [
    "https://www.python.org#content",
    "https://www.python.org#python-network",
    "https://www.python.org/",
    "https://www.python.org/psf/",
    "https://docs.python.org"
  ],
  "images_count": 1
}
{
  "url": "https://httpbin.org/html",
  "title": "",
  "h1": [
    "Herman Melville - Moby-Dick"
  ],
  "text_length": 3594,
  "links_count": 0,
  "links": [],
  "images_count": 0
}
{
  "url": "https://docs.python.org/3/",
  "title": "3.15.1 Documentation",
  "h1": [
    "Python 3.15.1 documentation"
  ],
  "text_length": 3217,
  "links_count": 9

## Тестирование 2 дня

In [16]:
VALID_HTML = """
<html>
  <head>
    <title>Тестовая страница</title>
    <meta name="description" content="Описание страницы">
  </head>
  <body>
    <h1>Главный заголовок</h1>
    <p>Основной текст страницы.</p>
    <img src="/logo.png" alt="Логотип">
    <table><tr><th>Город</th></tr><tr><td>Москва</td></tr></table>
    <ul><li>Чай</li><li>Кофе</li></ul>
  </body>
</html>
"""


async def test_parse_valid_html():
    result = await HTMLParser().parse_html(VALID_HTML, "https://example.com")
    assert result["title"] == "Тестовая страница", f"title: {result['title']}"
    assert result["metadata"]["description"] == "Описание страницы", "нет description"
    assert result["headings"]["h1"] == ["Главный заголовок"], "неверный h1"
    assert "Основной текст страницы." in result["text"], "нет текста"
    assert result["images"] == [
        {"src": "https://example.com/logo.png", "alt": "Логотип"}
    ], "неверные картинки"
    assert result["tables"] == [[["Город"], ["Москва"]]], "неверная таблица"
    assert result["lists"] == [{"type": "ul", "items": ["Чай", "Кофе"]}], "неверный список"


async def test_parse_broken_html():
    parser = HTMLParser()
    broken = "<title>Битая</title><h1>Заголовок<p>Текст<a href='/page'>ссылка<table><tr><td>A<td>B"
    result = await parser.parse_html(broken, "https://example.com")
    assert result["title"] == "Битая", f"title: {result['title']}"
    assert result["links"] == ["https://example.com/page"], f"links: {result['links']}"
    assert result["tables"] == [[["A", "B"]]], f"tables: {result['tables']}"

    result = await parser.parse_html(None, "https://example.com")
    assert result == parser.empty_result("https://example.com"), "ожидали пустой результат"


async def test_extract_links():
    html = """
    <a href="https://python.org">1</a>
    <a href="mailto:info@example.com">2</a>
    <a href="javascript:void(0)">3</a>
    <a href="tel:+70000000000">4</a>
    <a>без href</a>
    """
    soup = BeautifulSoup(html, "lxml")
    links = HTMLParser().extract_links(soup, "https://example.com")
    assert links == ["https://python.org"], f"links: {links}"


async def test_relative_urls():
    html = """
    <a href="/about">1</a>
    <a href="page2.html">2</a>
    <a href="../archive">3</a>
    <a href="//cdn.example.org/lib">4</a>
    """
    soup = BeautifulSoup(html, "lxml")
    links = HTMLParser().extract_links(soup, "https://example.com/news/today")
    assert links == [
        "https://example.com/about",
        "https://example.com/news/page2.html",
        "https://example.com/archive",
        "https://cdn.example.org/lib",
    ], f"links: {links}"


await run_tests([test_parse_valid_html, test_parse_broken_html, test_extract_links, test_relative_urls])

11:49:35 [WARNING] Не удалось разобрать HTML (TypeError): https://example.com


✅ test_parse_valid_html
✅ test_parse_broken_html
✅ test_extract_links
✅ test_relative_urls

Пройдено 4 из 4


## Демонстрация 3 дня

In [17]:
async def save_results(crawler: AsyncCrawler, path: str):
    data = {
        "processed": [
            {**page, "depth": crawler.url_depths[url]}
            for url, page in crawler.processed_urls.items()
        ],
        "failed": crawler.failed_urls,
    }
    async with aiofiles.open(path, "w", encoding="utf-8") as file:
        await file.write(json.dumps(data, ensure_ascii=False, indent=2))


async def demo_crawl():
    crawler = AsyncCrawler(max_concurrent=10, max_depth=2, per_domain_limit=3)
    logger.setLevel(logging.WARNING)
    try:
        results = await crawler.crawl(
            start_urls=[
                "https://quotes.toscrape.com/",
                "https://books.toscrape.com/",
            ],
            max_pages=150,
            same_domain_only=True,
        )
    finally:
        logger.setLevel(logging.INFO)
        await crawler.close()

    await save_results(crawler, "crawl_results.json")

    by_domain = Counter(urlparse(url).netloc for url in results)
    by_depth = Counter(crawler.url_depths[url] for url in results)
    print(f"\nОбработано: {len(results)} страниц")
    print(f"Ошибок: {len(crawler.failed_urls)}")
    print(f"По сайтам: {dict(by_domain)}")
    print(f"По глубине: {dict(sorted(by_depth.items()))}")
    print("Результаты сохранены в crawl_results.json")


await demo_crawl()

Обработано: 1 | В очереди: 37 | Ошибок: 0 | Заблокировано: 0 | Скорость: 1.0 стр/сек | Запросов: 5.0 req/sec | Средняя задержка: 0.00 сек
Обработано: 9 | В очереди: 104 | Ошибок: 0 | Заблокировано: 0 | Скорость: 4.5 стр/сек | Запросов: 6.0 req/sec | Средняя задержка: 0.00 сек
Обработано: 20 | В очереди: 146 | Ошибок: 0 | Заблокировано: 0 | Скорость: 6.7 стр/сек | Запросов: 7.7 req/sec | Средняя задержка: 0.00 сек
Обработано: 32 | В очереди: 162 | Ошибок: 0 | Заблокировано: 0 | Скорость: 8.0 стр/сек | Запросов: 8.7 req/sec | Средняя задержка: 0.00 сек
Обработано: 42 | В очереди: 161 | Ошибок: 0 | Заблокировано: 0 | Скорость: 8.4 стр/сек | Запросов: 9.4 req/sec | Средняя задержка: 0.00 сек
Обработано: 59 | В очереди: 313 | Ошибок: 0 | Заблокировано: 0 | Скорость: 9.8 стр/сек | Запросов: 11.4 req/sec | Средняя задержка: 0.00 сек
Обработано: 73 | В очереди: 494 | Ошибок: 0 | Заблокировано: 0 | Скорость: 10.4 стр/сек | Запросов: 12.8 req/sec | Средняя задержка: 0.00 сек
Обработано: 86 | В о

## Тестирование 3 дня

In [18]:
FAKE_SITE = {
    "https://site.com/": '<a href="/a">a</a><a href="/b">b</a><a href="https://other.com/x">x</a>',
    "https://site.com/a": '<a href="/a/deep">deep</a><a href="/">home</a><a href="/b">b</a>',
    "https://site.com/b": '<a href="/a">a</a><a href="/login">login</a>',
    "https://site.com/a/deep": '<a href="/a/deeper">deeper</a>',
    "https://site.com/a/deeper": "",
    "https://site.com/login": "",
    "https://other.com/x": "",
}


class FakeCrawler(AsyncCrawler):
    def __init__(self, site: dict | None = None, delays: dict | None = None, **kwargs):
        super().__init__(**kwargs)
        self.site = site or FAKE_SITE
        self.delays = delays or {}
        self.fetch_calls: list[str] = []

    async def fetch_url(self, url: str) -> str | None:
        self.fetch_calls.append(url)
        await asyncio.sleep(self.delays.get(url, 0.01))
        if url in self.site:
            return f"<html><body>{self.site[url]}</body></html>"
        self.failed_urls[url] = "NotFound"
        return None


async def run_fake_crawl(
    max_depth: int, site: dict | None = None, delays: dict | None = None, **filters
) -> FakeCrawler:
    crawler = FakeCrawler(site=site, delays=delays, max_concurrent=3, max_depth=max_depth)
    try:
        await crawler.crawl(["https://site.com/"], max_pages=100, **filters)
    finally:
        await crawler.close()
    return crawler


async def test_queue_priority():
    queue = CrawlerQueue()
    queue.add_url("https://site.com/a", priority=0)
    queue.add_url("https://site.com/b", priority=5)
    queue.add_url("https://site.com/c", priority=0)
    queue.add_url("https://site.com/d", priority=-1)
    queue.add_url("https://site.com/b", priority=10)

    order = [await queue.get_next() for _ in range(5)]
    assert order == [
        "https://site.com/b",
        "https://site.com/a",
        "https://site.com/c",
        "https://site.com/d",
        None,
    ], f"порядок: {order}"


async def test_depth_limit():
    crawler = await run_fake_crawl(max_depth=1)
    depths = [crawler.url_depths[url] for url in crawler.visited_urls]
    assert max(depths) == 1, f"глубины: {depths}"
    assert "https://site.com/a/deep" not in crawler.visited_urls, "зашли глубже max_depth"

    crawler = await run_fake_crawl(max_depth=2)
    assert "https://site.com/a/deep" in crawler.visited_urls, "не дошли до глубины 2"
    assert "https://site.com/a/deeper" not in crawler.visited_urls, "зашли на глубину 3"


async def test_url_filters():
    crawler = await run_fake_crawl(
        max_depth=3, same_domain_only=True, exclude_patterns=[r"/login"]
    )
    assert crawler.visited_urls == {
        "https://site.com/",
        "https://site.com/a",
        "https://site.com/b",
        "https://site.com/a/deep",
        "https://site.com/a/deeper",
    }, f"visited: {crawler.visited_urls}"

    crawler = await run_fake_crawl(max_depth=3, include_patterns=[r"/a"])
    assert crawler.visited_urls == {
        "https://site.com/",
        "https://site.com/a",
        "https://site.com/a/deep",
        "https://site.com/a/deeper",
    }, f"visited: {crawler.visited_urls}"


async def test_no_duplicates():
    crawler = await run_fake_crawl(max_depth=3)
    calls = crawler.fetch_calls
    assert len(calls) == len(set(calls)), f"повторные загрузки: {calls}"
    assert set(calls) == crawler.visited_urls, "visited_urls не совпадает с загрузками"
    assert len(crawler.visited_urls) == len(FAKE_SITE), f"посещено {len(crawler.visited_urls)}"

SHORTCUT_SITE = {
    "https://site.com/": '<a href="/slow">slow</a><a href="/fast1">fast1</a>',
    "https://site.com/slow": '<a href="/target">target</a>',
    "https://site.com/fast1": '<a href="/fast2">fast2</a>',
    "https://site.com/fast2": '<a href="/target">target</a>',
    "https://site.com/target": '<a href="/child">child</a>',
    "https://site.com/child": "",
}

async def test_shortest_depth():
    crawler = await run_fake_crawl(
        max_depth=3,
        site=SHORTCUT_SITE,
        delays={"https://site.com/slow": 0.5},
    )
    assert crawler.url_depths["https://site.com/target"] == 2, (
        f"глубина target: {crawler.url_depths['https://site.com/target']}"
    )
    assert "https://site.com/child" in crawler.visited_urls, "страница в пределах max_depth потеряна"

await run_tests([test_queue_priority, test_depth_limit, test_url_filters, test_no_duplicates, test_shortest_depth])


✅ test_queue_priority
Обработано: 4 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Скорость: 39.4 стр/сек | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
Обработано: 6 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Скорость: 59.1 стр/сек | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
✅ test_depth_limit
Обработано: 5 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Скорость: 49.8 стр/сек | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
Обработано: 4 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Скорость: 39.5 стр/сек | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
✅ test_url_filters
Обработано: 7 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Скорость: 69.1 стр/сек | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
✅ test_no_duplicates
Обработано: 6 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Скорость: 9.9 стр/сек | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
✅ test_shortest_depth

Пройдено 5 из 5


## Демонстрация 4 дня

In [19]:
async def demo_polite_crawl():
    crawler = AsyncCrawler(
        max_concurrent=5,
        max_depth=1,
        requests_per_second=2.0,
        respect_robots=True,
        min_delay=0.5,
        jitter=0.2,
        user_agent="MyBot/1.0",
    )
    logger.setLevel(logging.WARNING)
    start = time.perf_counter()
    try:
        results = await crawler.crawl(
            start_urls=[
                "https://docs.python.org/3/",
                "https://quotes.toscrape.com/",
            ],
            max_pages=50,
            same_domain_only=True,
        )
    finally:
        logger.setLevel(logging.INFO)
        await crawler.close()
    elapsed = time.perf_counter() - start

    rate_stats = crawler.rate_limiter.get_stats()
    by_domain = Counter(urlparse(url).netloc for url in results)
    print("\nИтоги")
    print(f"  Обработано страниц: {len(results)}")
    print(f"  По сайтам: {dict(by_domain)}")
    print(f"  Заблокировано robots.txt: {len(crawler.blocked_urls)}")
    print(f"  Ошибок: {len(crawler.failed_urls)}")
    print(f"  Время: {elapsed:.1f} сек")

    print("\nЗадержки")
    print(f"  Разрешено не чаще чем раз в: {crawler.rate_limiter.interval:.2f} сек (плюс до {crawler.jitter:.2f} случайно)")
    print(f"  Самая короткая фактическая пауза: {rate_stats['min_delay']:.2f} сек")
    print(f"  Средняя фактическая пауза: {rate_stats['avg_delay']:.2f} сек")

    print("\nЗаблокированные адреса")
    for url in sorted(crawler.blocked_urls):
        print(f"  {url}")


await demo_polite_crawl()

Обработано: 2 | В очереди: 89 | Ошибок: 0 | Заблокировано: 0 | Скорость: 2.0 стр/сек | Запросов: 3.0 req/sec | Средняя задержка: 0.68 сек
Обработано: 5 | В очереди: 87 | Ошибок: 0 | Заблокировано: 0 | Скорость: 2.5 стр/сек | Запросов: 2.9 req/sec | Средняя задержка: 0.75 сек
Обработано: 7 | В очереди: 85 | Ошибок: 0 | Заблокировано: 0 | Скорость: 2.3 стр/сек | Запросов: 2.6 req/sec | Средняя задержка: 0.87 сек
Обработано: 9 | В очереди: 83 | Ошибок: 0 | Заблокировано: 0 | Скорость: 2.2 стр/сек | Запросов: 2.4 req/sec | Средняя задержка: 0.91 сек
Обработано: 11 | В очереди: 81 | Ошибок: 0 | Заблокировано: 0 | Скорость: 2.2 стр/сек | Запросов: 2.3 req/sec | Средняя задержка: 0.91 сек
Обработано: 13 | В очереди: 79 | Ошибок: 0 | Заблокировано: 0 | Скорость: 2.2 стр/сек | Запросов: 2.0 req/sec | Средняя задержка: 0.91 сек
Обработано: 15 | В очереди: 76 | Ошибок: 0 | Заблокировано: 0 | Скорость: 2.1 стр/сек | Запросов: 2.2 req/sec | Средняя задержка: 0.90 сек
Обработано: 18 | В очереди: 74 

11:50:07 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.10/
11:50:07 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.9/
11:50:07 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.8/
11:50:07 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.7/
11:50:07 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.6/
11:50:07 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.5/


Обработано: 44 | В очереди: 42 | Ошибок: 0 | Заблокировано: 6 | Скорость: 2.5 стр/сек | Запросов: 2.8 req/sec | Средняя задержка: 0.80 сек

Итоги
  Обработано страниц: 44
  По сайтам: {'docs.python.org': 22, 'quotes.toscrape.com': 22}
  Заблокировано robots.txt: 6
  Ошибок: 0
  Время: 17.8 сек

Задержки
  Разрешено не чаще чем раз в: 0.50 сек (плюс до 0.20 случайно)
  Самая короткая фактическая пауза: 0.64 сек
  Средняя фактическая пауза: 0.80 сек

Заблокированные адреса
  https://docs.python.org/3.10/
  https://docs.python.org/3.5/
  https://docs.python.org/3.6/
  https://docs.python.org/3.7/
  https://docs.python.org/3.8/
  https://docs.python.org/3.9/


## Тестирование 4 дня

In [20]:
class FakeResponse:
    def __init__(self, status: int, text: str):
        self.status = status
        self._text = text

    async def text(self) -> str:
        return self._text

    async def __aenter__(self):
        return self

    async def __aexit__(self, *args):
        return False


class FakeSession:
    def __init__(self, pages: dict[str, str]):
        self.pages = pages
        self.calls: list[str] = []

    def get(self, url: str) -> FakeResponse:
        self.calls.append(url)
        if url in self.pages:
            return FakeResponse(200, self.pages[url])
        return FakeResponse(404, "")


class FakePoliteCrawler(AsyncCrawler):
    def __init__(self, site: dict[str, str], robots_pages: dict[str, str], **kwargs):
        super().__init__(**kwargs)
        self.site = site
        self.robots = RobotsParser(FakeSession(robots_pages))
        self.request_times: list[float] = []

    async def try_fetch(self, url: str) -> str:
        async with self.semaphores.acquire(url):
            await self.rate_limiter.acquire(urlparse(url).netloc)
            self.request_times.append(time.monotonic())
            if url in self.site:
                return f"<html><body>{self.site[url]}</body></html>"
            raise PermanentError("NotFound", url, 404)


def gaps_between(times: list[float]) -> list[float]:
    times = sorted(times)
    return [later - earlier for earlier, later in zip(times, times[1:])]


async def test_rate_limit_single_domain():
    limiter = RateLimiter(requests_per_second=10.0)
    times = []

    async def request():
        await limiter.acquire("a.com")
        times.append(time.monotonic())

    await asyncio.gather(*(request() for _ in range(5)))
    gaps = gaps_between(times)
    assert all(gap >= 0.09 for gap in gaps), f"паузы: {gaps}"


async def test_rate_limit_multiple_domains():
    domains = ["a.com", "a.com", "a.com", "b.com", "b.com", "b.com"]
    for per_domain, expected in ((True, 0.2), (False, 0.5)):
        limiter = RateLimiter(requests_per_second=10.0, per_domain=per_domain)
        start = time.monotonic()
        await asyncio.gather(*(limiter.acquire(domain) for domain in domains))
        elapsed = time.monotonic() - start
        assert expected - 0.02 <= elapsed <= expected + 0.05, (
            f"per_domain={per_domain}: {elapsed:.2f} сек вместо {expected}"
        )


ROBOTS_TXT = """
User-agent: *
Disallow: /private/
Crawl-delay: 2

User-agent: BadBot
Disallow: /

User-agent: SlowBot
Crawl-delay: 0.5
"""


async def test_robots_parsing():
    session = FakeSession({"https://site.com/robots.txt": ROBOTS_TXT})
    robots = RobotsParser(session)
    await robots.fetch_robots("https://site.com/page")
    info = await robots.fetch_robots("https://site.com/other")
    assert session.calls == ["https://site.com/robots.txt"], f"загрузок: {session.calls}"
    assert info["crawl_delay"] == 2.0, f"crawl_delay: {info['crawl_delay']}"
    assert robots.get_crawl_delay("SlowBot/1.0", "site.com") == 0.5, "дробный Crawl-delay не прочитан"
    assert robots.can_fetch("https://site.com/public") is True, "public должен быть разрешён"
    assert robots.can_fetch("https://site.com/private/data") is False, "private должен быть запрещён"
    assert robots.can_fetch("https://site.com/public", "BadBot") is False, "BadBot должен быть заблокирован"

    info = await robots.fetch_robots("https://empty.com/")
    assert info["allow_all"] is True, "без robots.txt всё разрешено"


POLITE_SITE = {
    "https://polite.com/": '<a href="/public">public</a><a href="/private/secret">secret</a>',
    "https://polite.com/public": "",
    "https://polite.com/private/secret": "",
}


async def test_blocked_urls():
    crawler = FakePoliteCrawler(
        site=POLITE_SITE,
        robots_pages={"https://polite.com/robots.txt": "User-agent: *\nDisallow: /private/\n"},
        respect_robots=True,
        max_depth=1,
    )
    try:
        results = await crawler.crawl(["https://polite.com/"])
    finally:
        await crawler.close()
    assert crawler.blocked_urls == {"https://polite.com/private/secret"}, f"blocked: {crawler.blocked_urls}"
    assert set(results) == {"https://polite.com/", "https://polite.com/public"}, f"results: {set(results)}"
    assert len(crawler.request_times) == 2, f"запросов: {len(crawler.request_times)}"


async def test_delays_respected():
    limiter = RateLimiter(requests_per_second=100.0, min_delay=0.2, jitter=0.1)
    times = []
    for _ in range(4):
        await limiter.acquire("a.com")
        times.append(time.monotonic())
    gaps = gaps_between(times)
    assert all(0.19 <= gap <= 0.32 for gap in gaps), f"min_delay и jitter: {gaps}"

    site = {f"https://polite.com/page{i}": "" for i in range(4)}
    crawler = FakePoliteCrawler(
        site=site,
        robots_pages={"https://polite.com/robots.txt": "User-agent: *\nCrawl-delay: 0.5\n"},
        respect_robots=True,
    )
    try:
        await crawler.fetch_urls(list(site))
    finally:
        await crawler.close()
    gaps = gaps_between(crawler.request_times)
    assert all(gap >= 0.49 for gap in gaps), f"Crawl-delay: {gaps}"


def make_site(domain: str, pages: int) -> dict[str, str]:
    links = "".join(f'<a href="/page{i}">{i}</a>' for i in range(pages))
    site = {f"https://{domain}/": links}
    site.update({f"https://{domain}/page{i}": "" for i in range(pages)})
    return site


async def test_domains_in_parallel():
    site = {**make_site("a.com", 6), **make_site("b.com", 6)}
    crawler = FakePoliteCrawler(
        site=site,
        robots_pages={},
        requests_per_second=2.0,
        max_concurrent=3,
        max_depth=1,
    )
    start = time.monotonic()
    try:
        results = await crawler.crawl(["https://a.com/", "https://b.com/"])
    finally:
        await crawler.close()
    elapsed = time.monotonic() - start
    assert len(results) == 14, f"обработано {len(results)}"
    assert elapsed < 4.0, f"сайты обходились по очереди: {elapsed:.1f} сек"


await run_tests([
    test_rate_limit_single_domain,
    test_rate_limit_multiple_domains,
    test_robots_parsing,
    test_blocked_urls,
    test_delays_respected,
    test_domains_in_parallel,
])

✅ test_rate_limit_single_domain


11:50:08 [WARNING] Заблокировано robots.txt: https://polite.com/private/secret


✅ test_rate_limit_multiple_domains
✅ test_robots_parsing
Обработано: 2 | В очереди: 0 | Ошибок: 0 | Заблокировано: 1 | Скорость: 19.9 стр/сек | Запросов: 2.0 req/sec | Средняя задержка: 0.00 сек
✅ test_blocked_urls
✅ test_delays_respected
Обработано: 4 | В очереди: 10 | Ошибок: 0 | Заблокировано: 0 | Скорость: 4.0 стр/сек | Запросов: 4.0 req/sec | Средняя задержка: 0.50 сек
Обработано: 8 | В очереди: 6 | Ошибок: 0 | Заблокировано: 0 | Скорость: 4.0 стр/сек | Запросов: 4.0 req/sec | Средняя задержка: 0.50 сек
Обработано: 12 | В очереди: 2 | Ошибок: 0 | Заблокировано: 0 | Скорость: 4.0 стр/сек | Запросов: 4.0 req/sec | Средняя задержка: 0.50 сек
Обработано: 14 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Скорость: 4.5 стр/сек | Запросов: 4.5 req/sec | Средняя задержка: 0.50 сек
✅ test_domains_in_parallel

Пройдено 6 из 6


## Демонстрация 5 дня

In [21]:
async def save_error_report(crawler: AsyncCrawler, path: str):
    report = {
        "statistics": crawler.get_error_stats(),
        "failed_urls": crawler.failed_urls,
        "circuit_breaker": {
            domain: crawler.circuit_breaker.get_state(domain)
            for domain in crawler.circuit_breaker.results
        },
    }
    async with aiofiles.open(path, "w", encoding="utf-8") as file:
        await file.write(json.dumps(report, ensure_ascii=False, indent=2))


async def demo_error_handling():
    strategy = RetryStrategy(
        max_retries=3,
        backoff_factor=2.0,
        retry_on=[TransientError, NetworkError],
        base_delay=0.5,
        retry_limits={NetworkError: 1},
    )
    breaker = CircuitBreaker(failure_rate=0.9, window_size=10, min_requests=4, recovery_timeout=30.0)
    crawler = AsyncCrawler(
        max_concurrent=5,
        read_timeout=1,
        timeout_growth=2.0,
        retry_strategy=strategy,
        circuit_breaker=breaker,
    )
    urls = [
        "https://httpbin.org/get",
        "https://httpbin.org/html",
        "https://httpbin.org/delay/2",
        "https://httpbin.org/status/429",
        "https://httpbin.org/status/500",
        "https://httpbin.org/status/404",
        "https://httpbin.org/status/403",
        "https://this-site-does-not-exist-12345.com",
        "https://httpstat.us/503",
        "https://httpstat.us/503?n=1",
        "https://httpstat.us/503?n=2",
        "https://httpstat.us/503?n=3",
    ]
    logger.setLevel(logging.WARNING)
    start = time.perf_counter()
    try:
        results = await crawler.fetch_urls(urls)
    finally:
        logger.setLevel(logging.INFO)
        await crawler.close()
    elapsed = time.perf_counter() - start

    await save_error_report(crawler, "error_report.json")

    stats = crawler.get_error_stats()
    print("\nИтоги")
    print(f"  Загружено: {len(results)} из {len(urls)} за {elapsed:.1f} сек")
    print(f"  Ошибки по типам: {stats['errors_by_type']}")
    print(f"  Всего повторов: {stats['retries']}, из них спасли запрос: {stats['successful_retries']}")
    print(f"  Среднее время на повтор: {stats['avg_retry_time']:.2f} сек")
    print("  Постоянные ошибки:")
    for url, reason in stats["permanent_urls"].items():
        print(f"    {url}: {reason}")
    print("Отчёт сохранён в error_report.json")


await demo_error_handling()

11:50:14 [ERROR] Попытка 1: TransientError (HTTP 500): https://httpbin.org/status/500
11:50:14 [WARNING] TransientError (HTTP 500) на попытке 1, повтор через 0.50 сек: https://httpbin.org/status/500
11:50:14 [ERROR] Попытка 1: NetworkError (ClientConnectorDNSError): https://this-site-does-not-exist-12345.com
11:50:14 [WARNING] NetworkError (ClientConnectorDNSError) на попытке 1, повтор через 0.50 сек: https://this-site-does-not-exist-12345.com
11:50:14 [ERROR] Попытка 1: TransientError (HTTP 429): https://httpbin.org/status/429
11:50:14 [WARNING] TransientError (HTTP 429) на попытке 1, повтор через 1.50 сек: https://httpbin.org/status/429
11:50:15 [ERROR] Попытка 1: TransientError (SocketTimeoutError): https://httpbin.org/delay/2
11:50:15 [WARNING] TransientError (SocketTimeoutError) на попытке 1, повтор через 0.50 сек: https://httpbin.org/delay/2
11:50:15 [ERROR] Попытка 1: PermanentError (HTTP 404): https://httpbin.org/status/404
11:50:15 [ERROR] Итог: неудача после 1 попыток (Perman


Итоги
  Загружено: 3 из 12 за 15.2 сек
  Ошибки по типам: {'TransientError': 15, 'NetworkError': 2, 'PermanentError': 2, 'CircuitOpenError': 4}
  Всего повторов: 14, из них спасли запрос: 1
  Среднее время на повтор: 3.29 сек
  Постоянные ошибки:
    https://httpbin.org/status/404: HTTP 404
    https://httpbin.org/status/403: HTTP 403
Отчёт сохранён в error_report.json


## Тестирование 5 дня

In [22]:
def http_error(status: int) -> aiohttp.ClientResponseError:
    return aiohttp.ClientResponseError(request_info=None, history=(), status=status)


class ScriptedResponse:
    def __init__(self, outcome):
        self.outcome = outcome
        self.status = outcome if isinstance(outcome, int) else 200

    async def __aenter__(self):
        if isinstance(self.outcome, Exception):
            raise self.outcome
        return self

    async def __aexit__(self, *args):
        return False

    def raise_for_status(self):
        if self.status >= 400:
            raise http_error(self.status)

    async def text(self) -> str:
        return "<html><body>ok</body></html>"


class ScriptedSession:
    def __init__(self, script: dict[str, list]):
        self.script = {url: list(outcomes) for url, outcomes in script.items()}
        self.calls: list[tuple[str, float]] = []

    def get(self, url: str, **kwargs) -> ScriptedResponse:
        self.calls.append((url, time.monotonic()))
        outcomes = self.script.get(url, [])
        outcome = outcomes.pop(0) if outcomes else 200
        return ScriptedResponse(outcome)

    def calls_for(self, url: str) -> list[float]:
        return [moment for called_url, moment in self.calls if called_url == url]


class ScriptedCrawler(AsyncCrawler):
    def __init__(self, script: dict[str, list], **kwargs):
        super().__init__(**kwargs)
        self.real_session = self.session
        self.session = ScriptedSession(script)

    async def close(self):
        await self.real_session.close()


def fast_strategy(**kwargs) -> RetryStrategy:
    return RetryStrategy(max_retries=3, base_delay=0.01, **kwargs)


async def run_scripted(script: dict[str, list], strategy: RetryStrategy) -> ScriptedCrawler:
    crawler = ScriptedCrawler(script, retry_strategy=strategy)
    try:
        await crawler.fetch_urls(list(script))
    finally:
        await crawler.close()
    return crawler


async def test_retry_on_timeout():
    url = "https://site.com/slow"
    crawler = await run_scripted(
        {url: [asyncio.TimeoutError(), asyncio.TimeoutError(), 200]}, fast_strategy()
    )
    attempts = len(crawler.session.calls_for(url))
    assert attempts == 3, f"попыток: {attempts}"
    assert url not in crawler.failed_urls, "страница должна загрузиться с третьей попытки"


async def test_retry_on_503():
    ok_url = "https://site.com/recovers"
    bad_url = "https://site.com/always-503"
    crawler = await run_scripted({ok_url: [503, 503, 200], bad_url: [503] * 10}, fast_strategy())
    assert len(crawler.session.calls_for(ok_url)) == 3, "503 должен повторяться до успеха"
    assert ok_url not in crawler.failed_urls, "страница должна загрузиться"
    attempts = len(crawler.session.calls_for(bad_url))
    assert attempts == 4, f"попыток: {attempts}"
    assert crawler.failed_urls[bad_url] == "TransientError: HTTP 503", crawler.failed_urls[bad_url]


async def test_no_retry_on_404():
    url = "https://site.com/missing"
    crawler = await run_scripted({url: [404] * 5}, fast_strategy())
    attempts = len(crawler.session.calls_for(url))
    assert attempts == 1, f"попыток: {attempts}"
    assert crawler.failed_urls[url] == "PermanentError: HTTP 404", crawler.failed_urls[url]


async def test_exponential_backoff():
    strategy = RetryStrategy(max_retries=3, backoff_factor=2.0, base_delay=0.05)
    delays = [round(strategy.get_delay(n), 6) for n in (1, 2, 3)]
    assert delays == [0.05, 0.1, 0.2], f"паузы по формуле: {delays}"
    assert round(strategy.get_delay(1, http_error(429)), 6) == 0.15, "для 429 пауза втрое длиннее"

    url = "https://site.com/flaky"
    crawler = await run_scripted({url: [503, 503, 503, 200]}, strategy)
    times = crawler.session.calls_for(url)
    gaps = [later - earlier for earlier, later in zip(times, times[1:])]
    for gap, expected in zip(gaps, [0.05, 0.1, 0.2]):
        assert expected <= gap < expected + 0.05, f"фактические паузы: {gaps}"


async def test_error_stats():
    script = {
        "https://site.com/ok": [200],
        "https://site.com/saved": [503, 200],
        "https://site.com/missing": [404],
        "https://site.com/forbidden": [403],
        "https://site.com/dead": [aiohttp.ClientConnectionError()] * 5,
    }
    crawler = ScriptedCrawler(script, retry_strategy=fast_strategy(retry_limits={NetworkError: 1}))
    try:
        await crawler.fetch_urls(list(script))
        await crawler.parser.parse_html(None, "https://site.com/broken")
    finally:
        await crawler.close()

    stats = crawler.get_error_stats()
    assert stats["errors_by_type"] == {
        "TransientError": 1,
        "PermanentError": 2,
        "NetworkError": 2,
        "ParseError": 1,
    }, stats["errors_by_type"]
    assert stats["retries"] == 2, f"повторов: {stats['retries']}"
    assert stats["successful_retries"] == 1, f"успешных повторов: {stats['successful_retries']}"
    assert stats["avg_retry_time"] > 0, "среднее время на повтор не посчитано"
    assert set(stats["permanent_urls"]) == {
        "https://site.com/missing",
        "https://site.com/forbidden",
    }, stats["permanent_urls"]


async def test_circuit_breaker():
    breaker = CircuitBreaker(failure_rate=0.5, window_size=4, min_requests=2, recovery_timeout=0.2)
    breaker.record_failure("site.com")
    breaker.record_failure("site.com")
    assert breaker.get_state("site.com") == "open", "домен должен заблокироваться"
    assert breaker.allow_request("site.com") is False, "запросы к заблокированному домену запрещены"

    breaker.record_success("site.com")
    assert breaker.get_state("site.com") == "open", "успех без пробы не должен снимать блокировку"

    await asyncio.sleep(0.25)
    assert breaker.allow_request("site.com") is True, "после паузы должна пройти проба"
    assert breaker.allow_request("site.com") is False, "одновременно проходит только одна проба"
    breaker.record_success("site.com")
    assert breaker.get_state("site.com") == "closed", "успешная проба снимает блокировку"


await run_tests([
    test_retry_on_timeout,
    test_retry_on_503,
    test_no_retry_on_404,
    test_exponential_backoff,
    test_error_stats,
    test_circuit_breaker,
])

11:50:29 [ERROR] Попытка 1: TransientError (TimeoutError): https://site.com/slow
11:50:29 [WARNING] TransientError (TimeoutError) на попытке 1, повтор через 0.01 сек: https://site.com/slow
11:50:29 [ERROR] Попытка 2: TransientError (TimeoutError): https://site.com/slow
11:50:29 [WARNING] TransientError (TimeoutError) на попытке 2, повтор через 0.02 сек: https://site.com/slow
11:50:29 [ERROR] Попытка 1: TransientError (HTTP 503): https://site.com/recovers
11:50:29 [WARNING] TransientError (HTTP 503) на попытке 1, повтор через 0.01 сек: https://site.com/recovers
11:50:29 [ERROR] Попытка 1: TransientError (HTTP 503): https://site.com/always-503
11:50:29 [WARNING] TransientError (HTTP 503) на попытке 1, повтор через 0.01 сек: https://site.com/always-503
11:50:29 [ERROR] Попытка 2: TransientError (HTTP 503): https://site.com/recovers
11:50:29 [WARNING] TransientError (HTTP 503) на попытке 2, повтор через 0.02 сек: https://site.com/recovers
11:50:29 [ERROR] Попытка 2: TransientError (HTTP 50

✅ test_retry_on_timeout
✅ test_retry_on_503
✅ test_no_retry_on_404


11:50:29 [ERROR] Попытка 3: TransientError (HTTP 503): https://site.com/flaky
11:50:29 [WARNING] TransientError (HTTP 503) на попытке 3, повтор через 0.20 сек: https://site.com/flaky
11:50:29 [ERROR] Попытка 1: TransientError (HTTP 503): https://site.com/saved
11:50:29 [WARNING] TransientError (HTTP 503) на попытке 1, повтор через 0.01 сек: https://site.com/saved
11:50:29 [ERROR] Попытка 1: PermanentError (HTTP 404): https://site.com/missing
11:50:29 [ERROR] Итог: неудача после 1 попыток (PermanentError: HTTP 404): https://site.com/missing
11:50:29 [ERROR] Попытка 1: PermanentError (HTTP 403): https://site.com/forbidden
11:50:29 [ERROR] Итог: неудача после 1 попыток (PermanentError: HTTP 403): https://site.com/forbidden
11:50:29 [ERROR] Попытка 1: NetworkError (ClientConnectionError): https://site.com/dead
11:50:29 [WARNING] NetworkError (ClientConnectionError) на попытке 1, повтор через 0.01 сек: https://site.com/dead
11:50:29 [ERROR] Попытка 2: NetworkError (ClientConnectionError): h

✅ test_exponential_backoff
✅ test_error_stats
✅ test_circuit_breaker

Пройдено 6 из 6
